# Fleet Maintenance Planning — Cost Improvement V4

Starting point: the 15,000-scoring FleetMaintainance notebook. Original features, risk models, repair models, eight-group kernel model, and baseline policy are retained. V4 adds two controlled local/global repair-residual weighting variants. The final policy is selected using the existing three-prior cross-validated cost comparison. Run the full notebook to measure whether either variant improves validation cost; no leaderboard improvement is assumed.

In [ ]:
from pathlib import Path
import sklearn

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold

SEED = 42
INSPECTION, SHORTAGE, SURPLUS = 10.0, 0.20, 0.02
SENSORS = [f"s{i:03d}" for i in range(170)]
SCHEMA = ["id", "flag", "budget"]

def find_data():
    required = {"train.csv", "test.csv", "sample_submission.csv"}
    kaggle = Path("/kaggle/input")
    candidates = [Path.cwd(), *(p.parent for p in sorted(kaggle.rglob("train.csv")))]
    matches = []
    for folder in dict.fromkeys(candidates):
        if all((folder / name).is_file() for name in required):
            test_cols = set(pd.read_csv(folder / "test.csv", nrows=0).columns)
            train_cols = set(pd.read_csv(folder / "train.csv", nrows=0).columns)
            if {"id", *SENSORS} <= test_cols and {"failure", "repair_cost", *SENSORS} <= train_cols:
                matches.append(folder)
    if len(matches) != 1:
        raise ValueError(f"Expected one P2 dataset. Matching directories: {matches}")
    return matches[0]

data = find_data()
train = pd.read_csv(data / "train.csv", dtype={"id": str})
test = pd.read_csv(data / "test.csv", dtype={"id": str})
sample = pd.read_csv(data / "sample_submission.csv", dtype={"id": str})
assert len(train) == 56_000 and len(test) == len(sample) == 20_000
assert list(sample) == SCHEMA
assert test.id.is_unique and sample.id.is_unique and set(test.id) == set(sample.id)
assert train.failure.isin([0, 1]).all()
failure = train.failure.to_numpy(dtype=int)
repair = train.repair_cost.fillna(0).to_numpy()
assert np.isfinite(repair).all() and np.array_equal(repair > 0, failure == 1)
print(f"{len(train):,} training vehicles, {failure.sum():,} recorded failures")
print(f"scikit-learn {sklearn.__version__}, NumPy {np.__version__}")

## Sensor readings

The test channels `s055`–`s063` match training channels `s054`–`s062` after a one-column shift. Scale them to a common resolution. Use stable counters to measure each reading relative to the vehicle’s overall sensor level.

In [ ]:
shift = {}
for name in SENSORS:
    old, new = train[name].dropna(), test[name].dropna()
    if min(old.nunique(), new.nunique()) < 2:
        shift[name] = 1.0
    else:
        distribution_shift = ks_2samp(old, new, method="asymp").statistic
        missingness_shift = abs(len(old) / len(train) - len(new) / len(test))
        shift[name] = max(distribution_shift, missingness_shift)

stable = [name for name in SENSORS if shift[name] < 0.10]

frequently_missing = train[stable].isna().mean().to_numpy() >= 0.20

def features(frame):
    readings = frame[stable].to_numpy(dtype="float32")
    missing = np.isnan(readings)
    counts = missing.sum(axis=1, dtype="float32")
    group_counts = missing[:, frequently_missing].sum(axis=1, dtype="float32")
    return np.column_stack((readings, counts, group_counts))

X, X_test = features(train), features(test)
print(f"Stable raw sensors: {len(stable)}")

# Normalize each reading by its training-depot scale before computing ratios.
scale = train[stable].quantile(0.90).to_numpy()
scale = np.where(scale > 0, scale, 1.0)
anchor_names = ["s071", "s123", "s149", "s053", "s012"]
anchor_cols = [stable.index(c) for c in anchor_names if c in stable]

# Adjacent columns match across depots after a one-column offset.
source_channels = [f"s{i:03d}" for i in range(54, 63)]
target_channels = [f"s{i:03d}" for i in range(55, 64)]
resolution = np.array([3, 10, 10, 100, 100, 100, 10, 10, 10], dtype="float32")
channel_scale = np.maximum(train[source_channels].quantile(0.9).to_numpy(dtype="float32"), 1)

def relative_features(frame, channels):
    raw = frame[stable].to_numpy(dtype="float32")
    scaled = raw / scale.astype("float32")
    exposure = pd.DataFrame(scaled[:, anchor_cols]).median(axis=1).to_numpy()
    log_readings = np.log1p(scaled)
    ratios = np.log1p(scaled / (0.01 + exposure[:, None]))
    parts = [log_readings, ratios, features(frame)[:, -2:]]
    # These blocks have the same missingness pattern and adjacent channels.
    for start, end in [(23, 33), (37, 47), (80, 90), (109, 119), (129, 139), (154, 164)]:
        cols = [stable.index(f"s{i:03d}") for i in range(start, end) if f"s{i:03d}" in stable]
        if len(cols) < 2:
            continue
        block = scaled[:, cols]
        total = np.sum(block, axis=1)
        parts.append(block / (0.01 + total[:, None]))
    block = frame[channels].to_numpy(dtype="float32")
    block = np.rint(block / resolution) * resolution / channel_scale
    parts.extend([np.log1p(block), np.log1p(block / (0.01 + exposure[:, None]))])
    return np.column_stack(parts).astype("float32")

Z, Z_test = relative_features(train, source_channels), relative_features(test, target_channels)

def active_channel_ratio(frame, name):
    active = frame[name].gt(100) & frame["s149"].gt(0)
    return frame.loc[active, name] / (1.0 + frame.loc[active, "s149"])

aligned_shift = [
    ks_2samp(active_channel_ratio(train, old), active_channel_ratio(test, new), method="asymp").statistic
    for old, new in zip(source_channels, target_channels)
]
print(f"Restored {len(aligned_shift)} channels. Largest active-ratio KS distance: {max(aligned_shift):.3f}")

## Failure risk and repair size

Short failure-free records receive less weight. Elastic net estimates how pairs of sensor readings affect the size of a repair.

In [ ]:
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.preprocessing import StandardScaler

def fit_predict(fit_idx, query, seed):
    x, y, c = X[fit_idx], failure[fit_idx], repair[fit_idx]
    positive = y == 1
    cls_options = dict(learning_rate=0.075, max_iter=210, max_bins=127,
                       early_stopping=False, random_state=seed)
    probabilities = []
    for leaves, min_leaf in ((7, 25), (15, 40)):
        clf = HistGradientBoostingClassifier(
            **cls_options, max_leaf_nodes=leaves, min_samples_leaf=min_leaf,
            l2_regularization=3.0, max_features=0.85)
        clf.fit(x, y)
        probabilities.append(clf.predict_proba(query)[:, 1])
    p = np.mean(probabilities, axis=0)

    mean_cost = c[positive].mean()
    weights = np.where(positive, c / mean_cost, 1.0)
    weighted = HistGradientBoostingClassifier(
        **cls_options, max_leaf_nodes=7, min_samples_leaf=35,
        l2_regularization=5.0, max_features=0.85)
    weighted.fit(x, y, sample_weight=weights)
    p_weighted = np.clip(weighted.predict_proba(query)[:, 1], 0, 0.985)
    expected_cost = mean_cost * p_weighted / (1 - p_weighted) * (1 - p)

    return np.column_stack((p, expected_cost))


def relative_predictions(indices, query, seed):
    x, y, c = Z[indices], failure[indices], repair[indices]
    positive = y == 1
    boost = HistGradientBoostingClassifier(max_leaf_nodes=12, min_samples_leaf=30, max_iter=175,
                                           max_bins=127, learning_rate=0.065, max_features=0.7,
                                           l2_regularization=6, early_stopping=False, random_state=seed)
    boost.fit(x, y, sample_weight=np.where(positive, 6.0, 1.0))
    score = boost.predict_proba(query)[:, 1]
    p_boost = score / (6.0 - 5.0 * score)

    # Sparse log-cost regression on the failed vehicles.
    def severity_features(rows):
        return np.column_stack((np.nan_to_num(rows, nan=0), np.isnan(rows), rows > 0))
    scaler = StandardScaler()
    design = scaler.fit_transform(severity_features(x[positive]))
    model = ElasticNet(alpha=0.09, l1_ratio=0.55, max_iter=4000)
    model.fit(design, np.log(c[positive]))
    log_cost = model.predict(scaler.transform(severity_features(query)))
    log_cost = np.clip(log_cost, np.log(c[positive].min()), np.log(c.max()))
    return np.column_stack((p_boost, log_cost))


from sklearn.kernel_approximation import Nystroem

def kernel_risk(indices, query, seed):
    # Fit the distance metric on the part of the training fleet with substantial sensor activity.
    x = Z[indices]
    y = failure[indices]
    active = x[:, stable.index("s071")]
    high = active >= np.nanquantile(active, 0.90)
    support = high | (y == 1)
    pool = np.nan_to_num(x[support], nan=-0.75)
    median = np.median(pool,axis=0)
    lo, hi = np.quantile(pool,[0.15,0.85],axis=0)
    spread = np.maximum(hi-lo,0.15)
    def standardized(raw):
        return np.clip((np.nan_to_num(raw,nan=-0.75) - median)/spread,-4,4)
    balanced = standardized(pool)
    # Prefer channels separating recorded failures from other high-activity vehicles.
    positive = y[support] == 1
    a, b = balanced[positive], balanced[~positive]
    separation = (a.mean(axis=0)-b.mean(axis=0))**2 / (a.var(axis=0)+b.var(axis=0)+0.02)
    select = np.argsort(separation)[-120:]
    def inputs(raw):
        return standardized(raw)[:,select]
    rng=np.random.default_rng(seed)
    difficult=np.flatnonzero(high & (y==0))
    landmarks=np.r_[np.flatnonzero(y==1),rng.choice(difficult,min(480,len(difficult)),replace=False)]
    # A low-rank RBF kernel keeps scoring all vehicles inexpensive.
    mapper=Nystroem(kernel="rbf",gamma=0.011,n_components=350,random_state=seed)
    mapper.fit(inputs(x[landmarks]))
    def transform(raw):
        return np.vstack([mapper.transform(inputs(raw[i:i+2500])) for i in range(0,len(raw),2500)])
    low=np.flatnonzero(~high & (y==0))
    sampled=rng.choice(low,min(len(low),6500),replace=False)
    rows=np.r_[np.flatnonzero(support),sampled]
    weights=np.where(y[rows]==1,8.0,np.where(high[rows],1.0,len(low)/len(sampled)))
    model=LogisticRegression(C=12,max_iter=350)
    model.fit(transform(x[rows]),y[rows],sample_weight=weights)
    s=model.predict_proba(transform(query))[:,1]
    return s/(8-7*s)

def channel_shapes(raw):
    parts = []
    # Each contiguous block is a set of readings with a shared acquisition pattern.
    for lo, hi in [(23,33), (37,47), (54,63), (80,90), (109,119), (129,139), (154,164)]:
        cols = [(stable + source_channels).index(f"s{i:03d}") for i in range(lo,hi)]
        block = raw[:,cols]
        total = block.sum(axis=1,keepdims=True)
        share = block / np.maximum(total,1.)
        left = np.cumsum(share,axis=1)[:,:-1]
        right = np.cumsum(share[:,::-1],axis=1)[:,-2::-1]
        ordinal = np.arange(len(cols)) / (len(cols)-1)
        mean = share @ ordinal
        variance = np.maximum(0,share @ (ordinal**2)-mean**2)
        summary = np.column_stack((mean, np.sqrt(variance), (share**2).sum(axis=1), share.max(axis=1)))
        parts.extend([summary,left,np.log1p(1000*left),np.log1p(1000*right)])
    return np.column_stack(parts)

# Read channel magnitudes before normalizing by the fleet-wide 90th percentile.
def cost_readings(frame, channels):
    raw = frame[stable].to_numpy(dtype=np.float64)
    matched = frame[channels].to_numpy(dtype=np.float64)
    return np.column_stack((raw, matched))

def cost_basis(raw, positive_scale):
    # The first transform retains low positive readings in heavy-tailed channels.
    logged = np.log1p(raw)
    relative = raw / positive_scale
    parts = [logged, np.log1p(10 * relative), np.arcsinh(relative), np.sqrt(np.clip(relative,0,25))]
    for lo, hi in [(23,33), (37,47), (54,63), (80,90), (109,119), (129,139), (154,164)]:
        cols = [(stable + source_channels).index(f"s{i:03d}") for i in range(lo,hi)]
        block = raw[:,cols]
        total = block.sum(axis=1, keepdims=True)
        share = block / np.maximum(total, 1.)
        parts.extend([share, np.sqrt(share), np.log1p(100*share)])
    return np.column_stack(parts)

def fit_cost_models(raw_fit, z_fit, costs, raw_query, z_query):
    positive_scale = np.nanquantile(raw_fit, 0.75, axis=0)
    positive_scale = np.where(np.isfinite(positive_scale) & (positive_scale > 1), positive_scale, 1)
    def design(raw, z):
        parts = [z, np.isnan(z), z > 0, cost_basis(raw, positive_scale), channel_shapes(raw)]
        return np.nan_to_num(np.column_stack(parts), nan=0).astype(np.float64)

    scaler = StandardScaler()
    x = scaler.fit_transform(design(raw_fit, z_fit))
    log_cost = np.log(costs)
    first = ElasticNet(alpha=0.075, l1_ratio=0.8, max_iter=5000).fit(x, log_cost)
    strength = np.abs(first.coef_)
    penalty = np.maximum(np.sqrt(strength / max(strength.max(), 1e-12)), 0.09)
    second = ElasticNet(alpha=0.035, l1_ratio=0.75, max_iter=5000).fit(x * penalty, log_cost)

    # Both regressions use the same sensor transformations.
    coefficients = np.column_stack((first.coef_, second.coef_ * penalty))
    intercepts = np.array([first.intercept_, second.intercept_])
    predictions = []
    for start in range(0, len(raw_query), 2500):
        batch = slice(start, start + 2500)
        values = scaler.transform(design(raw_query[batch], z_query[batch]))
        predictions.append(values @ coefficients + intercepts)
    return np.clip(np.vstack(predictions), np.log(costs.min()), np.log(costs.max()))

from scipy.linalg import cho_factor, cho_solve
from scipy.spatial.distance import cdist
from scipy.optimize import minimize

sensor_blocks = [(23, 33), (37, 47), (54, 63), (80, 90), (109, 119), (129, 139), (154, 164)]
cost_columns = stable + source_channels
bin_indices = [[cost_columns.index(f"s{i:03d}") for i in range(lo, hi)] for lo, hi in sensor_blocks]

def utilization_features(raw):
    total = raw[:, bin_indices[0]].sum(axis=1)
    reference_names = [cost_columns[j] for j in bin_indices[0]]
    typical = train[reference_names].sum(axis=1, min_count=len(reference_names)).median()
    sensor_scale = np.maximum(train[cost_columns].quantile(0.9).to_numpy(), 1)
    relative = raw / sensor_scale
    pieces = [np.log1p(relative), np.log1p(relative / (0.001 + total[:, None] / typical))]
    for cols in bin_indices:
        block = raw[:, cols]
        mass = block.sum(axis=1, keepdims=True)
        share = block / np.maximum(mass, 1)
        pieces.extend([share, np.cumsum(share, axis=1)[:, :-1], np.log1p(mass)])
    return np.column_stack(pieces).astype("float32")

def utilization_score(features, fit, query, seed):
    days = train.followup_days.to_numpy()[fit]
    y = failure[fit]
    # A short failure-free record is weaker evidence about annual risk.
    exposure = np.clip(days / 365.0, 0.08, 1.0)
    weights = np.where(y == 1, 6.0, exposure ** 2)
    model = HistGradientBoostingClassifier(
        max_leaf_nodes=10, learning_rate=0.055, max_iter=240, min_samples_leaf=35,
        max_bins=191, max_features=0.7, l2_regularization=7,
        early_stopping=False, random_state=seed)
    model.fit(features[fit], y, sample_weight=weights)
    return model.predict_proba(query)[:, 1]

def severity_groups():
    groups = np.array([next((k for k, (lo, hi) in enumerate(sensor_blocks)
                             if lo <= int(name[1:]) < hi), len(sensor_blocks))
                       for name in cost_columns])
    shares = np.concatenate([np.full(hi-lo, k) for k, (lo, hi) in enumerate(sensor_blocks) if lo != 54])
    relative = np.r_[groups[:len(stable)], groups[:len(stable)], 7, 7, shares, groups[-9:], groups[-9:]]
    return np.concatenate([np.tile(relative, 3), np.tile(groups, 4),
        *[np.full(3*(hi-lo), k) for k,(lo,hi) in enumerate(sensor_blocks)],
        *[np.full(4+3*(hi-lo-1), k) for k,(lo,hi) in enumerate(sensor_blocks)]])

def kernel_cost(raw, z, costs, raw_query, z_query):
    scales = np.maximum(np.nanquantile(raw, 0.75, axis=0), 1)
    def design(r, features):
        return np.nan_to_num(np.column_stack((
            features, np.isnan(features), features > 0, cost_basis(r, scales), channel_shapes(r))), nan=0)
    scaler = StandardScaler()
    x = scaler.fit_transform(design(raw, z))
    y = np.log(costs)
    first = ElasticNet(alpha=0.065, l1_ratio=0.8, max_iter=5000).fit(x, y)
    adaptive = np.maximum(np.sqrt(abs(first.coef_) / max(abs(first.coef_).max(), 1e-8)), 0.075)
    second = ElasticNet(alpha=0.031, l1_ratio=0.75, max_iter=5000).fit(x*adaptive, y)
    coefficients = second.coef_ * adaptive
    group = severity_groups()
    loadings = np.column_stack([coefficients * (group == k) for k in range(8)])
    active = np.flatnonzero(abs(coefficients) > 0.014)
    def coordinates(values):
        return values @ loadings, np.clip(values[:, active], -4, 4) * np.sqrt(abs(coefficients[active]))
    pooled, detailed = coordinates(x)
    def distances(a, b):
        return cdist(a, pooled, "sqeuclidean"), cdist(b, detailed, "sqeuclidean"), a @ pooled.T
    def covariance(params, distances):
        d1, d2, linear = distances
        v = np.exp(params)
        return v[0]*linear + v[1]*np.exp(-d1/v[2]) + v[3]*np.exp(-d2/v[4])
    train_dist = distances(pooled, detailed)
    identity = np.eye(len(y))
    centered = y - y.mean()
    def factor(params):
        matrix = covariance(params, train_dist) + np.exp(params[5])*identity
        return cho_factor(matrix, lower=True)
    def objective(params):
        chol = factor(params)
        return (0.5*centered @ cho_solve(chol, centered) + np.log(np.diag(chol[0])).sum()
                + 0.3*np.sum(params[:5]**2))
    start = np.log([1, .3, 4, .3, 6, .5])
    bounds = np.log([[.08,4],[.005,4],[.15,40],[.005,4],[.2,80],[.23,1.5]])
    params = minimize(objective, start, bounds=bounds, method="L-BFGS-B", options={"maxiter":45}).x
    alpha = cho_solve(factor(params), centered)
    preds = []
    for start in range(0, len(raw_query), 1500):
        batch = slice(start, start+1500)
        query = scaler.transform(design(raw_query[batch], z_query[batch]))
        pred = y.mean() + covariance(params, distances(*coordinates(query))) @ alpha
        preds.append(np.clip(pred, y.min(), y.max()))
    return np.concatenate(preds)

def interaction_cost(raw, z, costs, raw_query, z_query):
    scales = np.maximum(np.nanquantile(raw, 0.75, axis=0), 1)
    def design(readings, summaries):
        columns = (summaries, np.isnan(summaries), summaries > 0,
                   cost_basis(readings, scales), channel_shapes(readings))
        return np.nan_to_num(np.column_stack(columns), nan=0)

    scaler = StandardScaler()
    x = scaler.fit_transform(design(raw, z))
    y = np.log(costs)
    first = ElasticNet(alpha=0.061, l1_ratio=0.75, max_iter=5000).fit(x, y)
    strength = np.abs(first.coef_)
    weights = np.maximum(np.sqrt(strength / max(strength.max(), 1e-8)), 0.065)

    # Correlated versions of one sensor would otherwise dominate the pairwise terms.
    chosen = []
    for j in np.argsort(-strength):
        if strength[j] <= 0.004 or len(chosen) >= 26:
            break
        if not chosen or np.max(np.abs(x[:, chosen].T @ x[:, j])) / len(x) < 0.86:
            chosen.append(j)
    left, right = np.triu_indices(len(chosen), 1)
    pair_scaler = StandardScaler()

    def interactions(values):
        selected = np.clip(values[:, chosen], -3.5, 3.5)
        return np.column_stack((np.abs(selected), selected[:, left] * selected[:, right]))

    combined = x * weights
    if chosen:
        pairs = pair_scaler.fit_transform(interactions(x))
        combined = np.column_stack((combined, 0.5 * pairs))
    second = ElasticNet(alpha=0.032, l1_ratio=0.75, max_iter=5000).fit(combined, y)
    main_coef = second.coef_[:x.shape[1]] * weights
    pair_coef = second.coef_[x.shape[1]:] * 0.5
    predictions = []
    for start in range(0, len(raw_query), 1500):
        batch = slice(start, start + 1500)
        q = scaler.transform(design(raw_query[batch], z_query[batch]))
        prediction = q @ main_coef
        if chosen:
            prediction += pair_scaler.transform(interactions(q)) @ pair_coef
        predictions.append(prediction + second.intercept_)
    return np.clip(np.concatenate(predictions), y.min(), y.max())

## Inspection and parts budgets

Estimate the range of repair bills from failures with similar risk and severity scores. A shortage costs ten times as much as a return, so the parts order uses the 10/11 quantile of demand, including vehicles with zero demand.

In [ ]:
from scipy.special import expit, logit

def estimate_prior(indices):
    """Fit a two-class mixture to high-risk tails of stable sensors."""
    old = X[indices, :len(stable)]
    new = X_test[:, :len(stable)]
    y = failure[indices].astype(bool)
    candidates = []
    for col, name in enumerate(stable):
        if shift[name] >= 0.06:
            continue
        positive, negative = old[y, col], old[~y, col]
        if np.isfinite(positive).sum() < 0.85 * y.sum():
            continue
        direction = 1 if np.nanmedian(positive) >= np.nanmedian(negative) else -1
        cutoffs = np.nanquantile(direction * negative, [0.95, 0.97, 0.98, 0.99])
        positive_tail = (direction * positive[:, None] > cutoffs).mean(axis=0)
        negative_tail = (direction * negative[:, None] > cutoffs).mean(axis=0)
        test_tail = (direction * new[:, col, None] > cutoffs).mean(axis=0)
        contrast = positive_tail - negative_tail
        if contrast[0] >= 0.55:
            estimate = np.median((test_tail - negative_tail) / np.maximum(contrast, 1e-6))
            candidates.append((contrast[0], name, estimate))
    # Use the strongest class separators and a median to limit the effect of shifted sensors.
    top = sorted(candidates, reverse=True)[:12]
    if len(top) < 3:
        return y.mean(), top
    return float(np.clip(np.median([estimate for _, _, estimate in top]), 0.002, 0.15)), top


def prior_ratio(source, target):
    return (target / (1 - target)) / (source / (1 - source))


def vehicle_weights(actual, target, source):
    return np.where(actual > 0, target / source, (1 - target) / (1 - source))


def individual_cost(actual, flag, budget):
    return np.where(flag, INSPECTION + SHORTAGE * np.maximum(actual - budget, 0)
                    + SURPLUS * np.maximum(budget - actual, 0), actual)


def risk_inputs(pred):
    # Each column is a held-out failure score. Combine the scores on the log-odds scale.
    scores = np.clip(pred[["raw_risk", "relative_risk", "kernel_risk"]].to_numpy(), 1e-6, 1 - 1e-6)
    logits = logit(scores)
    average = logits[:, [0, 1]].mean(axis=1)
    return np.column_stack((logits, np.maximum(average + 4.5, 0), np.maximum(average + 1.5, 0)))


def calibrated_risk(ref, query, actual, target_rate):
    a, b = risk_inputs(ref), risk_inputs(query)
    obs = actual > 0
    model = LogisticRegression(C=0.5, max_iter=300)
    model.fit(a, obs)
    recorded = model.predict_proba(b)[:, 1]
    recorded_target = expit(logit(recorded) + np.log(prior_ratio(obs.mean(), target_rate)))
    return recorded_target


def shape_demand(ref, query, actual, use_kernel=False):
    observed = actual > 0
    y = np.log(actual[observed])
    def severity(pred):
        sparse = pred[["sparse_cost", "shape_cost", "adaptive_cost"]].to_numpy() @ [0.15, 0.425, 0.425]
        if use_kernel:
            return 0.5 * sparse + 0.5 * pred.kernel_cost.to_numpy()
        return sparse
    x = severity(ref[observed])
    slope = np.clip(((x-x.mean()) @ (y-y.mean()) + 12) / ((x-x.mean()) @ (x-x.mean()) + 12),.65,1.6)
    center = y.mean() + slope*(severity(query)-x.mean())
    errors = y - y.mean() - slope*(x-x.mean())
    return np.exp(np.clip(center,-4,11)), np.sort(np.exp(errors)), actual.max()


def parts_decisions(p, demand, expected_raw, raw_to_target, direct_share):
    scale, residuals, ceiling = demand
    prefix = np.r_[0,np.cumsum(residuals)]
    n = len(residuals)
    cap = np.searchsorted(residuals, ceiling / scale, side="right")
    mean = (scale * prefix[cap] + (n - cap)*ceiling) / n
    q = 1 - SURPLUS / ((SHORTAGE + SURPLUS) * np.maximum(p, 1e-12))
    index = np.clip(np.ceil(n*q).astype(int)-1, 0, n-1)
    budget = np.where(q>0,np.minimum(scale*residuals[index],ceiling),0.)
    below = np.searchsorted(residuals, budget/scale, side="right")
    returned = np.maximum((budget*below - scale*prefix[below])/n,0)
    shortage = np.maximum(mean - budget + returned,0)
    inspect = INSPECTION + SHORTAGE*p*shortage + SURPLUS*((1-p)*budget + p*returned)
    expected = p*mean
    direct = np.clip(raw_to_target*expected_raw, expected/3, 3*expected)
    flag = (1-direct_share)*expected + direct_share*direct > inspect
    return flag.astype(int), np.where(flag,np.round(budget,2),0)


def incumbent_risk(ref, query, actual, prior):
    def boost_logits(frame):
        score = 0.45 * frame.raw_risk.to_numpy() + 0.55 * frame.relative_risk.to_numpy()
        return logit(np.clip(score, 1e-7, 1 - 1e-7)).reshape(-1, 1)

    calibration = LogisticRegression(C=1).fit(boost_logits(ref), actual > 0)
    source_probability = calibration.predict_proba(boost_logits(query))[:, 1]
    odds = prior_ratio((actual > 0).mean(), prior)
    target_probability = expit(logit(source_probability) + np.log(odds))
    stacked = calibrated_risk(ref, query, actual, prior)
    risk = expit(logit(0.7 * stacked + 0.3 * target_probability) + np.log(1.15))
    return risk, source_probability, odds


def incumbent_decision(ref, query, actual, prior):
    risk, source_probability, _ = incumbent_risk(ref, query, actual, prior)
    demand = shape_demand(ref, query, actual)
    correction = risk / np.maximum(source_probability, 1e-7)
    return parts_decisions(risk, demand, query.direct_cost.to_numpy(), correction, 0.15)


def budget_inputs(budget, center):
    return np.column_stack((
        np.ones(len(budget)),
        np.clip(np.log(center / 350.0), -2, 2),
        np.clip(budget / center - 1.6, -1.5, 1.5),
    ))

def learn_budget_factors(budget, center, cost, prior):
    active = budget > 0
    x = budget_inputs(budget[active], center[active])
    target = cost[active] / budget[active]
    # A unit of multiplier changes the order by the original budget in cost units.
    weights = vehicle_weights(cost, prior, (cost > 0).mean())[active] * budget[active]
    weights /= weights.sum()
    quantile = SHORTAGE / (SHORTAGE + SURPLUS)
    penalty = np.array([0.03, 0.045, 0.045])

    def objective(coef):
        error = target - (1 + x @ coef)
        radius = np.hypot(error, 0.06)
        loss = (quantile - 0.5) * error + 0.5 * radius
        gradient = x.T @ (weights * (0.5 - quantile - 0.5 * error / radius))
        return weights @ loss + 0.5 * penalty @ (coef ** 2), gradient + penalty * coef

    fit = minimize(objective, np.zeros(3), method="L-BFGS-B", jac=True,
                   bounds=[(-0.45, 0.55), (-0.35, 0.35), (-0.35, 0.35)])
    if not fit.success or not np.isfinite(fit.x).all():
        raise ValueError(f"Budget calibration failed: {fit.message}")
    return fit.x

def apply_correction(budget, center, coef, ceiling):
    change = np.clip(budget_inputs(budget, center) @ coef, -0.4, 0.8)
    return np.minimum(ceiling, budget * (1 + change)).round(2)

def maintenance_risk(ref, query, actual, prior):
    original_risk, source_probability, odds = incumbent_risk(ref, query, actual, prior)

    def inputs(frame):
        activity = logit(np.clip(frame.utilization_score.to_numpy(), 1e-8, 1 - 1e-8))
        return np.column_stack((risk_inputs(frame), activity))

    model = LogisticRegression(C=0.5, max_iter=300).fit(inputs(ref), actual > 0)
    activity_risk = expit(model.decision_function(inputs(query)) + np.log(1.15 * odds))
    risk = 0.6 * original_risk + 0.4 * activity_risk
    return risk, source_probability


def proposal(ref, query, actual, prior):
    risk, source_probability = maintenance_risk(ref, query, actual, prior)
    demand = shape_demand(ref, query, actual, use_kernel=True)
    correction = risk / np.maximum(source_probability, 1e-7)
    flags, budgets = parts_decisions(risk, demand, query.direct_cost.to_numpy(), correction, 0.15)
    return flags, budgets, demand[0]


def adjusted_decision(ref, query, actual, prior, estimate=proposal):
    training_budgets = np.empty(len(actual))
    training_centers = np.empty(len(actual))
    folds = StratifiedKFold(3, shuffle=True, random_state=77)
    for fit, val in folds.split(ref, actual > 0):
        _, training_budgets[val], training_centers[val] = estimate(
            ref.iloc[fit], ref.iloc[val], actual[fit], prior)
    coefficients = learn_budget_factors(training_budgets, training_centers, actual, prior)
    flags, budgets, centers = estimate(ref, query, actual, prior)
    return flags, apply_correction(budgets, centers, coefficients, actual.max())

def repair_score(frame):
    baseline = frame[["sparse_cost", "shape_cost", "adaptive_cost", "kernel_cost"]].to_numpy()
    return 0.7 * (baseline @ [0.075, 0.2125, 0.2125, 0.5]) + 0.3 * frame.interaction_cost.to_numpy()


def joint_risk_inputs(frame):
    scores = risk_inputs(frame)
    activity = logit(np.clip(frame.utilization_score.to_numpy(), 1e-7, 1 - 1e-7))
    severity = frame[["adaptive_cost", "kernel_cost"]].mean(axis=1).to_numpy()
    level = np.clip(severity - np.log(200), -3, 3)
    direct = np.log(np.maximum(frame.direct_cost.to_numpy(), 1e-5))
    return np.column_stack((scores, activity, level, level * np.clip((scores[:, 0] + 4) / 4, -1, 1),
                            np.clip(direct - scores[:, 0] - severity, -4, 4)))


def local_proposal(ref, query, actual, prior):
    previous_risk, source = maintenance_risk(ref, query, actual, prior)
    observed = actual > 0
    odds = prior_ratio(observed.mean(), prior)
    model = LogisticRegression(C=0.35, max_iter=400).fit(joint_risk_inputs(ref), observed)
    calibrated = expit(model.decision_function(joint_risk_inputs(query)) + np.log(1.15 * odds))
    risk = 0.35 * previous_risk + 0.65 * calibrated

    failures = ref[observed]
    log_cost = np.log(actual[observed])
    scores = repair_score(failures)
    centered = scores - scores.mean()
    slope = np.clip((centered @ (log_cost - log_cost.mean()) + 12) / (centered @ centered + 12), 0.65, 1.6)
    location = np.exp(np.clip(log_cost.mean() + slope * (repair_score(query) - scores.mean()), -4, 11))
    errors = np.exp(log_cost - log_cost.mean() - slope * centered)
    order = np.argsort(errors)
    errors = errors[order]

    def coordinates(frame):
        return np.column_stack((repair_score(frame), risk_inputs(frame)[:, :3].mean(axis=1),
                                frame.kernel_cost.to_numpy() - frame.sparse_cost.to_numpy()))

    reference = coordinates(failures)[order]
    scale = np.maximum(reference.std(axis=0), 0.15) * [0.68, 1.1, 1.6]
    center = reference.mean(axis=0)
    reference = (reference - center) / scale
    positions = (coordinates(query) - center) / scale
    flags = np.empty(len(query), dtype=int)
    budgets = np.empty(len(query))
    for start in range(0, len(query), 500):
        batch = slice(start, start + 500)
        distance = cdist(positions[batch], reference, "sqeuclidean")
        weights = np.exp(-0.5 * np.minimum(distance, 80))
        # Keep fleet-wide repair experience in neighborhoods with few failures.
        weights += 25 / len(errors)
        weights /= weights.sum(axis=1, keepdims=True)
        weights = 0.85 * weights + 0.15 / len(errors)
        costs = np.minimum(location[batch, None] * errors, actual.max())
        p = risk[batch]
        quantile = 1 - SURPLUS / ((SHORTAGE + SURPLUS) * np.maximum(p, 1e-12))
        ix = np.minimum((np.cumsum(weights, axis=1) < quantile[:, None]).sum(axis=1), len(errors) - 1)
        parts = np.where(quantile > 0, costs[np.arange(len(p)), ix], 0)
        expected = p * (costs * weights).sum(axis=1)
        direct = query.direct_cost.to_numpy()[batch] * p / np.maximum(source[batch], 1e-7)
        skip = 0.85 * expected + 0.15 * np.clip(direct, expected / 3, 3 * expected)
        shortage = (np.maximum(costs - parts[:, None], 0) * weights).sum(axis=1)
        surplus = (np.maximum(parts[:, None] - costs, 0) * weights).sum(axis=1)
        inspect = INSPECTION + SHORTAGE * p * shortage + SURPLUS * ((1 - p) * parts + p * surplus)
        flags[batch] = skip > inspect
        budgets[batch] = np.where(flags[batch], parts.round(2), 0)
    return flags, budgets, location


def maintenance_decision(ref, query, actual, prior):
    return adjusted_decision(ref, query, actual, prior, estimate=local_proposal)

# Controlled alternatives: change only local/global shrinkage of repair residual weights.
def local_proposal_variant(ref, query, actual, prior, global_mix=0.30):
    previous_risk, source = maintenance_risk(ref, query, actual, prior)
    observed = actual > 0
    odds = prior_ratio(observed.mean(), prior)
    model = LogisticRegression(C=0.35, max_iter=400).fit(joint_risk_inputs(ref), observed)
    calibrated = expit(model.decision_function(joint_risk_inputs(query)) + np.log(1.15 * odds))
    risk = 0.35 * previous_risk + 0.65 * calibrated
    failures = ref[observed]
    log_cost = np.log(actual[observed])
    scores = repair_score(failures)
    centered = scores - scores.mean()
    slope = np.clip((centered @ (log_cost - log_cost.mean()) + 12) /
                    (centered @ centered + 12), 0.65, 1.6)
    location = np.exp(np.clip(log_cost.mean() + slope *
                              (repair_score(query) - scores.mean()), -4, 11))
    errors = np.exp(log_cost - log_cost.mean() - slope * centered)
    order = np.argsort(errors)
    errors = errors[order]

    def coordinates(frame):
        return np.column_stack((repair_score(frame),
                                risk_inputs(frame)[:, :3].mean(axis=1),
                                frame.kernel_cost.to_numpy() - frame.sparse_cost.to_numpy()))

    reference = coordinates(failures)[order]
    scale = np.maximum(reference.std(axis=0), 0.15) * [0.68, 1.1, 1.6]
    center = reference.mean(axis=0)
    reference = (reference - center) / scale
    positions = (coordinates(query) - center) / scale
    flags, budgets = np.empty(len(query), dtype=int), np.empty(len(query))
    for start in range(0, len(query), 500):
        batch = slice(start, start + 500)
        distance = cdist(positions[batch], reference, "sqeuclidean")
        weights = np.exp(-0.5 * np.minimum(distance, 80))
        weights += 25 / len(errors)
        weights /= weights.sum(axis=1, keepdims=True)
        weights = (1.0 - global_mix) * weights + global_mix / len(errors)
        costs = np.minimum(location[batch, None] * errors, actual.max())
        p = risk[batch]
        quantile = 1 - SURPLUS / ((SHORTAGE + SURPLUS) * np.maximum(p, 1e-12))
        ix = np.minimum((np.cumsum(weights, axis=1) < quantile[:, None]).sum(axis=1),
                        len(errors) - 1)
        parts = np.where(quantile > 0, costs[np.arange(len(p)), ix], 0)
        expected = p * (costs * weights).sum(axis=1)
        direct = query.direct_cost.to_numpy()[batch] * p / np.maximum(source[batch], 1e-7)
        skip = 0.85 * expected + 0.15 * np.clip(direct, expected / 3, 3 * expected)
        shortage = (np.maximum(costs - parts[:, None], 0) * weights).sum(axis=1)
        surplus = (np.maximum(parts[:, None] - costs, 0) * weights).sum(axis=1)
        inspect = INSPECTION + SHORTAGE * p * shortage + SURPLUS * ((1-p) * parts + p * surplus)
        flags[batch] = skip > inspect
        budgets[batch] = np.where(flags[batch], parts.round(2), 0)
    return flags, budgets, location


def maintenance_decision_global_10(ref, query, actual, prior):
    estimate = lambda a, b, c, d: local_proposal_variant(a, b, c, d, global_mix=0.10)
    return adjusted_decision(ref, query, actual, prior, estimate=estimate)


def maintenance_decision_global_30(ref, query, actual, prior):
    estimate = lambda a, b, c, d: local_proposal_variant(a, b, c, d, global_mix=0.30)
    return adjusted_decision(ref, query, actual, prior, estimate=estimate)


## Validation

Compare maintenance cost per vehicle at three assumed new-depot failure rates.

In [ ]:
raw_cost = cost_readings(train, source_channels)
test_cost = cost_readings(test, target_channels)
activity_train = utilization_features(raw_cost)
activity_test = utilization_features(test_cost)

names = ["raw_risk", "direct_cost", "relative_risk", "sparse_cost",
         "kernel_risk", "shape_cost", "adaptive_cost", "utilization_score", "kernel_cost", "interaction_cost"]
indices = np.arange(len(train))
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED+1).split(indices, failure))
oof = pd.DataFrame(np.nan, index=indices, columns=names)
test_folds = []
for fold, (fit, val) in enumerate(folds, 1):
    seed = SEED + 1 + fold
    pos = fit[failure[fit] == 1]
    z_query = np.vstack((Z[val], Z_test))
    raw_query = np.vstack((raw_cost[val], test_cost))
    predictions = np.column_stack((
        fit_predict(fit, np.vstack((X[val], X_test)), seed),
        relative_predictions(fit, z_query, seed),
        kernel_risk(fit, z_query, SEED+8+fold),
        fit_cost_models(raw_cost[pos], Z[pos], repair[pos], raw_query, z_query),
        utilization_score(activity_train, fit, np.vstack((activity_train[val], activity_test)), seed),
        kernel_cost(raw_cost[pos], Z[pos], repair[pos], raw_query, z_query),
        interaction_cost(raw_cost[pos], Z[pos], repair[pos], raw_query, z_query),
    ))
    oof.iloc[val] = predictions[:len(val)]
    test_folds.append(predictions[len(val):])
    print(f"Finished fold {fold}/{len(folds)}", flush=True)
test_scores = pd.DataFrame(np.mean(test_folds,axis=0), columns=names)
assert np.isfinite(oof.to_numpy()).all() and np.isfinite(test_scores.to_numpy()).all()


prior, _ = estimate_prior(indices)
policies = {
    "personal best": incumbent_decision,
    "previous revision": adjusted_decision,
    "local repair costs (baseline)": maintenance_decision,
    "local residuals 10% global": maintenance_decision_global_10,
    "local residuals 30% global": maintenance_decision_global_30,
}
losses = np.empty((len(train), len(policies)))
for fit, val in folds:
    ref, query = oof.iloc[fit], oof.iloc[val]
    fold_prior, _ = estimate_prior(fit)
    for j, make_decision in enumerate(policies.values()):
        flag, budget = make_decision(ref, query, repair[fit], fold_prior)
        losses[val, j] = individual_cost(repair[val], flag, budget)

rows = []
for rate in prior * np.array([0.65, 1.0, 1.35]):
    weights = vehicle_weights(repair, rate, failure.mean())
    rows.append([rate, *np.average(losses, axis=0, weights=weights)])
comparison = pd.DataFrame(rows, columns=["assumed failure rate", *policies])
print(comparison.round(4).to_string(index=False))

## Kaggle submission

Write the selected policy to `submission.csv` and print its SHA-256 fingerprint.

In [ ]:
from hashlib import sha256

# Choose among policies evaluated on the same validation folds and prior scenarios.
policy_scores = comparison[list(policies)].mean(axis=0)
selected_policy_name = policy_scores.idxmin()
selected_policy = policies[selected_policy_name]
flag, budget = selected_policy(oof, test_scores, repair, prior)
print(f"Submission policy selected by mean validation cost: {selected_policy_name}")
print(policy_scores.sort_values().round(5).to_string())
predictions = pd.DataFrame({"id": test.id, "flag": flag, "budget": budget}).set_index("id")
submission = predictions.loc[sample.id].reset_index()

output_dir = Path("/kaggle/working")
if not output_dir.is_dir():
    output_dir = Path.cwd()
output = output_dir / "submission.csv"
submission.to_csv(output, index=False)
written = pd.read_csv(output, dtype={"id": str})
assert written.columns.tolist() == SCHEMA
assert len(written) == 20_000 and written.id.is_unique and written.id.equals(sample.id)
assert written.flag.isin([0, 1]).all()
assert np.isfinite(written.budget).all() and written.budget.ge(0).all()
assert written.loc[written.flag.eq(0), "budget"].eq(0).all()
print(f"Saved {len(written):,} rows to {output}")
print(f"Inspections: {written.flag.sum():,}. Positive budgets: {written.budget.gt(0).sum():,}.")
for label, previous_policy in policies.items():
    old_flag, old_budget = previous_policy(oof, test_scores, repair, prior)
    changes = (flag != old_flag) | (budget != old_budget)
    print(f"Rows differing from {label}: {changes.sum():,}")
print(f"SHA-256: {sha256(output.read_bytes()).hexdigest()}")